# 13 — Version 2 Logistic Regression

**Owners:** Nanda / Khishan

This remains the transparent linear baseline. It receives the finalized Version 2
behavioural features but keeps training-only imputation, scaling, rare grouping,
frequency encoding and balanced loss.


## What Version 2 changes—and why

Version 1 remains our reproducible baseline. Version 2 adds behaviour that the
winning Kaggle solution showed was valuable, but implements it in a stricter
real-time form:

- `D` values are normalized against transaction day to expose stable date anchors.
- a conservative `uid_proxy` describes a possible client without using it as a label;
- counts, time since previous use, amount history and unique-value history describe
  behaviour;
- every historical feature uses only earlier transactions; and
- no feature reads `isFraud`, later rows, validation labels, or test labels.

The newest 15% remains the final test period. It is never used for feature or
hyperparameter selection.


In [ ]:
from pathlib import Path
_start = Path.cwd().resolve()
for _candidate in [_start, *_start.parents]:
    if (_candidate / "requirements-training.txt").exists():
        _requirements = _candidate / "requirements-training.txt"
        break
else:
    raise FileNotFoundError("Open this notebook from inside the cloned repository")
%pip install -q -r {_requirements}


In [ ]:
from pathlib import Path
import gc, json, os, sys, time
import numpy as np
import pandas as pd

def locate_project_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / ".git").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Run this notebook from inside the cloned repository")

PROJECT_ROOT = locate_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

V2_DATA_DIR = PROJECT_ROOT / "data" / "processed" / "v2"
V2_ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "v2"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Project root:", PROJECT_ROOT)
print("Version 2 data:", V2_DATA_DIR)
print("Version 2 artifacts:", V2_ARTIFACT_ROOT)


In [ ]:
required = [V2_DATA_DIR / name for name in ["train.parquet", "validation.parquet", "test.parquet"]]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run 10_v2_behavioral_data_preparation.ipynb first. Missing: " + ", ".join(missing)
    )

train, validation, test = [pd.read_parquet(path) for path in required]
FAST_RUN = False  # Only for code checks. Never report FAST_RUN metrics.
if FAST_RUN:
    def debug_sample(frame, rows):
        return (frame.groupby("isFraud", group_keys=False)
                .apply(lambda group: group.sample(
                    n=max(1, round(rows * len(group) / len(frame))),
                    random_state=RANDOM_SEED), include_groups=True)
                .sort_values(["TransactionDT", "TransactionID"]).reset_index(drop=True))
    train = debug_sample(train, 60_000)
    validation = debug_sample(validation, 20_000)
    test = debug_sample(test, 20_000)

TARGET = "isFraud"
DROP_FROM_MODEL = ["isFraud", "TransactionID"]
X_train, y_train = train.drop(columns=DROP_FROM_MODEL), train[TARGET].astype("int8")
X_validation, y_validation = validation.drop(columns=DROP_FROM_MODEL), validation[TARGET].astype("int8")
X_test, y_test = test.drop(columns=DROP_FROM_MODEL), test[TARGET].astype("int8")
development = pd.concat([train, validation], ignore_index=True)
print("Train:", X_train.shape, "fraud rate:", f"{y_train.mean():.4%}")
print("Validation:", X_validation.shape, "fraud rate:", f"{y_validation.mean():.4%}")
print("Test:", X_test.shape, "fraud rate:", f"{y_test.mean():.4%}")


In [ ]:
MODEL_KEY = "logistic_regression"

from datetime import datetime, timezone
from src.fraud_pipeline.artifacts import build_manifest, package_versions, write_json
from src.fraud_pipeline.evaluation import evaluate_binary_classifier, select_operating_threshold

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = V2_ARTIFACT_ROOT / MODEL_KEY / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("This Version 2 run will be saved to:", RUN_DIR)


## Fit the sparse pipeline

Logistic regression cannot learn complex interactions as naturally as boosted
trees, so its purpose is interpretability and a defensible baseline—not winning
the leaderboard.


In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from src.fraud_pipeline.preprocessing import build_logistic_preprocessor, infer_feature_groups

groups = infer_feature_groups(X_train, low_cardinality_max=100)
preprocessor = build_logistic_preprocessor(groups, rare_min_count=20)
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        C=0.1, penalty="l2", solver="saga", class_weight="balanced",
        max_iter=800, tol=0.001, n_jobs=-1, random_state=RANDOM_SEED, verbose=1,
    )),
])
started = time.perf_counter()
pipeline.fit(X_train, y_train)
training_seconds = time.perf_counter() - started
classifier = pipeline.named_steps["classifier"]
if int(classifier.n_iter_[0]) >= classifier.max_iter:
    raise RuntimeError("Logistic Regression reached max_iter; increase it before accepting this run")
print("Converged at iteration:", int(classifier.n_iter_[0]))


## Validation, threshold, and final test


In [ ]:
validation_probability = pipeline.predict_proba(X_validation)[:, 1]
threshold_record = select_operating_threshold(y_validation, validation_probability, minimum_precision=0.10)
threshold = float(threshold_record["threshold"])
validation_metrics = evaluate_binary_classifier(y_validation, validation_probability, threshold)
started = time.perf_counter()
test_probability = pipeline.predict_proba(X_test)[:, 1]
prediction_seconds = time.perf_counter() - started
test_metrics = evaluate_binary_classifier(y_test, test_probability, threshold)
display(pd.DataFrame([validation_metrics, test_metrics], index=["validation", "test"])[
    ["pr_auc", "roc_auc", "precision", "recall", "f1", "brier_score"]])


## Save, reload, and package


In [ ]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
coefficients = pipeline.named_steps["classifier"].coef_[0]
importance = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
importance["absolute_coefficient"] = importance.coefficient.abs()
importance.sort_values("absolute_coefficient", ascending=False).head(150).to_csv(
    RUN_DIR / "top_coefficients.csv", index=False)
joblib.dump(pipeline, RUN_DIR / "model.joblib", compress=3)
pd.DataFrame({"TransactionID": validation.TransactionID, "isFraud": y_validation,
              "probability": validation_probability}).to_parquet(RUN_DIR / "validation_predictions.parquet", index=False)
pd.DataFrame({"TransactionID": test.TransactionID, "isFraud": y_test,
              "probability": test_probability}).to_parquet(RUN_DIR / "test_predictions.parquet", index=False)
write_json(RUN_DIR / "threshold.json", threshold_record)
write_json(RUN_DIR / "metrics.json", {"validation": validation_metrics, "test": test_metrics})
write_json(RUN_DIR / "feature_schema.json", {"groups": groups,
           "behavioral_contract": "data/processed/v2/behavioral_contract.json"})
write_json(RUN_DIR / "training_config.json", {
    "model": "v2_logistic_regression", "run_id": RUN_ID, "fast_run": FAST_RUN,
    "random_seed": RANDOM_SEED, "training_seconds": training_seconds,
    "test_prediction_seconds": prediction_seconds,
    "converged_iteration": int(classifier.n_iter_[0]),
    "parameters": classifier.get_params(),
    "versions": package_versions(["numpy", "pandas", "scikit-learn", "joblib"]),
})
loaded = joblib.load(RUN_DIR / "model.joblib")
np.testing.assert_allclose(validation_probability[:5], loaded.predict_proba(X_validation.iloc[:5])[:, 1], rtol=1e-6, atol=1e-8)


In [ ]:
import shutil
write_json(RUN_DIR / "manifest.json", build_manifest(RUN_DIR))
archive_base = RUN_DIR.parent / f"{MODEL_KEY}_{RUN_ID}"
archive_path = Path(shutil.make_archive(str(archive_base), "gztar", root_dir=RUN_DIR))
print("Reload check passed.")
print("Artifact folder:", RUN_DIR)
print("Share this archive:", archive_path)
